In [26]:
from functools import partial
from typing import Dict, List, Optional, Tuple
import os
import pickle

import numpy as np

import jax
import jax.numpy as jnp
from flax import nnx

# Notes On Physical Assumptions (for dataset generation)

This notebook generates training data using a **center-biased spatial prior** and a **coarse-grained bead-size prior**.

## Important correction about bead size
Using MARTINI's common bead size directly for "4 residues per bead" is **not** correct.
MARTINI's classic mapping is roughly 4 heavy atoms per bead (not 4 residues per bead), and its LJ sigma is a force-field interaction length, not a direct residue-clump radius.

## Why center-biased point picking?
We model a giant simulation box where positions near the center are more probable than near edges.
Implementation: a 3D Gaussian centered at the box midpoint, then clipped to box bounds.

## Why 1-5 residues per bead?
We sample bead granularity uniformly from 1, 2, 3, 4, 5 residues per bead.
To convert residues-per-bead to bead radius we use residue-volume-like scaling:

$$r(n) = r_{1}\, n^{1/3}$$

where:
- $n$ = residues represented by a bead
- $r_1$ = effective radius for a 1-residue bead (default in code: $r_1 = 3.2\ \AA$)

This gives:
- 1 residue/bead -> 3.20 A
- 2 residues/bead -> 4.03 A
- 3 residues/bead -> 4.62 A
- 4 residues/bead -> 5.08 A
- 5 residues/bead -> 5.47 A

These values are much more physically plausible for residue-scale coarse graining than 2.35 A at 4 residues/bead.

## Rotational invariance idea
For each sampled pair of points in two spheres, we compute:
1. the raw Euclidean distance
2. a rotation-minimized lower bound that would be reachable by rotating points around sphere centers

We then use the smaller of those two distances when checking crosslink feasibility.
This captures your "if rotation can make points closer, use that" request.

In [34]:
class CrosslinkForwardModel:
    """
    JAX-accelerated dataset generator for lysine crosslink scenarios.

    This class is intentionally over-commented for users coming from NumPy/CPU code.
    Think of it as a bridge from familiar vectorized NumPy ideas to JAX+JIT.
    """

    def __init__(
        self,
        L: float = 21.0,
        seed: int = 0,
        prefer_gpu: bool = True,
        box_min: Tuple[float, float, float] = (0.0, 0.0, 0.0),
        box_max: Tuple[float, float, float] = (300.0, 300.0, 300.0),
        center_bias_sigma_fraction: float = 0.22,
        residue_reference_radius_angstrom: float = 3.2,
    ):
        # Crosslinker cutoff in Angstrom. A sampled pair is a "success" when d < L.
        self.L = float(L)

        # Simulation box corners.
        self.box_min = jnp.asarray(box_min, dtype=jnp.float32)
        self.box_max = jnp.asarray(box_max, dtype=jnp.float32)

        # Controls how concentrated the center-biased prior is.
        # Smaller value => stronger concentration near the box center.
        self.center_bias_sigma_fraction = float(center_bias_sigma_fraction)

        # Effective radius of a 1-residue coarse bead in Angstrom.
        # Default 3.2 A is a residue-scale geometric choice.
        self.residue_reference_radius_angstrom = float(residue_reference_radius_angstrom)

        # PRNG key is JAX's random state; always split before use.
        self.key = jax.random.PRNGKey(seed)

        # Device selection: accelerator first (mps/gpu/tpu), CPU fallback.
        self.device = self._select_device(prefer_gpu=prefer_gpu)

        # Keep both rich configs and compact numeric rows.
        self.configurations: List[Dict] = []
        self.results: List[List[float]] = []

        # ----- Bead-size prior: 1 to 5 residues per bead -----
        # IMPORTANT:
        # We do NOT use MARTINI sigma to map residues/bead sizes.
        # MARTINI's classic mapping is roughly 4 heavy atoms per bead, not 4 residues per bead.
        # Here we use residue-volume-like scaling r(n) = r1 * n^(1/3).
        self.residue_levels = jnp.asarray([1, 2, 3, 4, 5], dtype=jnp.int32)
        self.bead_radii_angstrom = self._compute_bead_radii(
            self.residue_levels, self.residue_reference_radius_angstrom
        )

    @staticmethod
    def _select_device(prefer_gpu: bool = True):
        devices = jax.devices()
        if prefer_gpu:
            accel = [d for d in devices if d.platform != "cpu"]
            if accel:
                return accel[0]
        cpu = [d for d in devices if d.platform == "cpu"]
        return cpu[0] if cpu else devices[0]

    @staticmethod
    def _split_key(key: jax.Array) -> Tuple[jax.Array, jax.Array]:
        # Sequential 2-way splits are more reliable here than Python-side unpacking
        # of jax.random.split(key, n) into multiple variables.
        return jax.random.split(key, 2)

    @staticmethod
    def _compute_bead_radii(residue_levels: jax.Array, r1: float) -> jax.Array:
        # Residue-based geometric scaling (volume-like): radius grows as n^(1/3).
        return float(r1) * (residue_levels.astype(jnp.float32) ** (1.0 / 3.0))

    @staticmethod
    @partial(jax.jit, static_argnums=(1,))
    def _sample_center_biased_centers(
        key: jax.Array,
        num_points: int,
        box_min: jax.Array,
        box_max: jax.Array,
        sigma_fraction: float,
    ) -> jax.Array:
        center = 0.5 * (box_min + box_max)
        sigma = (box_max - box_min) * sigma_fraction

        raw = center + sigma * jax.random.normal(key, (num_points, 3))
        clipped = jnp.clip(raw, box_min, box_max)
        return clipped

    @staticmethod
    @partial(jax.jit, static_argnums=(1,))
    def _sample_bead_levels_and_radii(
        key: jax.Array,
        num_configs: int,
        residue_levels: jax.Array,
        bead_radii: jax.Array,
    ) -> Tuple[jax.Array, jax.Array]:
        idx = jax.random.randint(key, shape=(num_configs,), minval=0, maxval=residue_levels.shape[0])
        levels = residue_levels[idx]
        radii = bead_radii[idx]
        return levels, radii

    @staticmethod
    @partial(jax.jit, static_argnums=(3,))
    def _sample_points_inside_sphere(
        key: jax.Array,
        center: jax.Array,
        radius: float,
        n_points: int,
    ) -> jax.Array:
        key_dir, key_r = jax.random.split(key)
        direction = jax.random.normal(key_dir, (n_points, 3))
        direction = direction / (jnp.linalg.norm(direction, axis=1, keepdims=True) + 1e-12)

        u = jax.random.uniform(key_r, (n_points, 1))
        radial = radius * (u ** (1.0 / 3.0))
        return center[None, :] + radial * direction

    @staticmethod
    @partial(jax.jit, static_argnums=(6, 7))
    def _mc_probability_with_rotation(
        key: jax.Array,
        c1: jax.Array,
        c2: jax.Array,
        r1: float,
        r2: float,
        L: float,
        n_trials: int,
        use_rotation_minimization: bool,
    ) -> jax.Array:
        key1, key2 = jax.random.split(key)
        q1 = CrosslinkForwardModel._sample_points_inside_sphere(key1, c1, r1, n_trials)
        q2 = CrosslinkForwardModel._sample_points_inside_sphere(key2, c2, r2, n_trials)

        raw_dist = jnp.linalg.norm(q1 - q2, axis=1)

        center_dist = jnp.linalg.norm(c1 - c2)
        radial_1 = jnp.linalg.norm(q1 - c1[None, :], axis=1)
        radial_2 = jnp.linalg.norm(q2 - c2[None, :], axis=1)
        rot_min_dist = jnp.maximum(0.0, center_dist - radial_1 - radial_2)

        effective_dist = jnp.where(
            use_rotation_minimization,
            jnp.minimum(raw_dist, rot_min_dist),
            raw_dist,
        )

        return jnp.mean(effective_dist < L)

    @staticmethod
    @partial(jax.jit, static_argnums=(6, 7))
    def _mc_probability_batch(
        keys: jax.Array,
        centers1: jax.Array,
        centers2: jax.Array,
        radii1: jax.Array,
        radii2: jax.Array,
        L: float,
        n_trials: int,
        use_rotation_minimization: bool,
    ) -> jax.Array:
        fn = lambda k, c1, c2, r1, r2: CrosslinkForwardModel._mc_probability_with_rotation(
            k, c1, c2, r1, r2, L, n_trials, use_rotation_minimization
        )
        return jax.vmap(fn)(keys, centers1, centers2, radii1, radii2)

    def generate_configurations(
        self,
        num_configs: int,
        n_trials: int = 50000,
        batch_size: int = 64,
        use_rotation_minimization: bool = True,
        verbose: bool = True,
    ) -> None:
        """
        Generate dataset rows with features and target probability.

        Features per row:
        [center_distance, radius1, radius2, crosslinker_length, residues_per_bead1, residues_per_bead2]
        Target:
        [crosslink_probability]
        """
        if verbose:
            print(f"Device selected: {self.device}")
            print(
                "Sampling center-biased sphere centers, bead sizes (1..5 residues), "
                "and Monte Carlo crosslink probabilities..."
            )

        # Avoid Python-side unpacking of multi-way key splits.
        self.key, kc1 = self._split_key(self.key)
        self.key, kc2 = self._split_key(self.key)
        centers1 = self._sample_center_biased_centers(
            kc1, int(num_configs), self.box_min, self.box_max, self.center_bias_sigma_fraction
        )
        centers2 = self._sample_center_biased_centers(
            kc2, int(num_configs), self.box_min, self.box_max, self.center_bias_sigma_fraction
        )

        self.key, kr1 = self._split_key(self.key)
        self.key, kr2 = self._split_key(self.key)
        levels1, radii1 = self._sample_bead_levels_and_radii(
            kr1, int(num_configs), self.residue_levels, self.bead_radii_angstrom
        )
        levels2, radii2 = self._sample_bead_levels_and_radii(
            kr2, int(num_configs), self.residue_levels, self.bead_radii_angstrom
        )

        centers1 = jax.device_put(centers1, self.device)
        centers2 = jax.device_put(centers2, self.device)
        radii1 = jax.device_put(radii1, self.device)
        radii2 = jax.device_put(radii2, self.device)
        levels1 = jax.device_put(levels1, self.device)
        levels2 = jax.device_put(levels2, self.device)

        n_cfg = int(num_configs)
        n_batches = (n_cfg + batch_size - 1) // batch_size
        self.key, kbatch = self._split_key(self.key)
        batch_roots = jax.random.split(kbatch, n_batches)

        probs_list = []
        for b in range(n_batches):
            start = b * batch_size
            end = min((b + 1) * batch_size, n_cfg)
            bsz = end - start

            keys = jax.random.split(batch_roots[b], bsz)
            probs_b = self._mc_probability_batch(
                keys,
                centers1[start:end],
                centers2[start:end],
                radii1[start:end],
                radii2[start:end],
                self.L,
                int(n_trials),
                bool(use_rotation_minimization),
            )
            probs_list.append(probs_b)

            if verbose and ((b + 1) % max(1, n_batches // 10) == 0):
                print(f"Processed batch {b + 1}/{n_batches}")

        probabilities = jnp.concatenate(probs_list, axis=0)

        centers1_np = np.asarray(jax.device_get(centers1))
        centers2_np = np.asarray(jax.device_get(centers2))
        radii1_np = np.asarray(jax.device_get(radii1))
        radii2_np = np.asarray(jax.device_get(radii2))
        levels1_np = np.asarray(jax.device_get(levels1))
        levels2_np = np.asarray(jax.device_get(levels2))
        probs_np = np.asarray(jax.device_get(probabilities), dtype=np.float32)

        center_dist_np = np.linalg.norm(centers1_np - centers2_np, axis=1).astype(np.float32)

        self.configurations = []
        self.results = []

        for i in range(n_cfg):
            cfg = {
                "center1": centers1_np[i],
                "center2": centers2_np[i],
                "radius1": float(radii1_np[i]),
                "radius2": float(radii2_np[i]),
                "residues_per_bead1": int(levels1_np[i]),
                "residues_per_bead2": int(levels2_np[i]),
                "center_distance": float(center_dist_np[i]),
                "L": float(self.L),
                "probability": float(probs_np[i]),
            }
            self.configurations.append(cfg)
            self.results.append(
                [
                    cfg["center_distance"],
                    cfg["radius1"],
                    cfg["radius2"],
                    cfg["L"],
                    float(cfg["residues_per_bead1"]),
                    float(cfg["residues_per_bead2"]),
                    cfg["probability"],
                ]
            )

        if verbose:
            print(f"Completed: generated {n_cfg} configurations.")

    def save_training_data(self, filename: str = "surrogate_model_data", fmt: str = "hdf5") -> None:
        """Save dataset to disk.

        X columns are:
        [center_distance, radius1, radius2, L, residues_per_bead1, residues_per_bead2]
        y column is:
        [probability]
        """
        if not self.results:
            raise ValueError("No data to save. Run generate_configurations() first.")

        arr = np.asarray(self.results, dtype=np.float32)
        X_data = arr[:, :-1]
        y_data = arr[:, -1:]

        if fmt == "npz":
            np.savez_compressed(f"{filename}.npz", X=X_data, y=y_data)
            print(f"Saved training data to {filename}.npz")
        elif fmt == "hdf5":
            import h5py
            with h5py.File(f"{filename}.h5", "w") as f:
                f.create_dataset("X", data=X_data, compression="gzip")
                f.create_dataset("y", data=y_data, compression="gzip")
            print(f"Saved training data to {filename}.h5")
        else:
            raise ValueError("fmt must be 'npz' or 'hdf5'")

    def prepare_training_data(self, test_size: float = 0.2, save_scaler: bool = True):
        """Prepare CPU NumPy arrays for ML workflows (with standard scaling)."""
        if not self.results:
            raise ValueError("No data to prepare. Run generate_configurations() first.")

        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import StandardScaler

        arr = np.asarray(self.results, dtype=np.float32)
        X_all = arr[:, :-1]
        y_all = arr[:, -1:]

        X_train, X_val, y_train, y_val = train_test_split(
            X_all, y_all, test_size=test_size, random_state=42
        )

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
        X_val_scaled = scaler.transform(X_val).astype(np.float32)

        if save_scaler:
            with open("data_scaler.pkl", "wb") as f:
                pickle.dump(scaler, f)
            print("Saved scaler to data_scaler.pkl")

        return X_train_scaled, X_val_scaled, y_train.astype(np.float32), y_val.astype(np.float32), scaler

    def to_jax_training_data(self, test_size: float = 0.2, save_scaler: bool = True):
        """Prepare and move train/validation arrays to the selected JAX device."""
        X_train, X_val, y_train, y_val, scaler = self.prepare_training_data(
            test_size=test_size, save_scaler=save_scaler
        )

        X_train_j = jax.device_put(jnp.asarray(X_train), self.device)
        X_val_j = jax.device_put(jnp.asarray(X_val), self.device)
        y_train_j = jax.device_put(jnp.asarray(y_train), self.device)
        y_val_j = jax.device_put(jnp.asarray(y_val), self.device)

        return X_train_j, X_val_j, y_train_j, y_val_j, scaler

In [37]:
# Smoke test for the new physically motivated generator
estimator = CrosslinkForwardModel(
    L=21.0,
    seed=42,
    prefer_gpu=True,
    box_min=(0.0, 0.0, 0.0),
    box_max=(300.0, 300.0, 300.0),
    center_bias_sigma_fraction=0.20,
)

print(f"Selected device: {estimator.device}")
print("Bead radii table (A):")
for n, r in zip(estimator.residue_levels.tolist(), estimator.bead_radii_angstrom.tolist()):
    print(f"  {n} residue/bead -> radius {r:.3f} A")

estimator.generate_configurations(
    num_configs=5000,
    n_trials=2000,
    batch_size=32,
    use_rotation_minimization=True,
    verbose=True,
)

print(f"Generated rows: {len(estimator.results)}")
print("First row [center_d, r1, r2, L, res1, res2, prob]:")
print(estimator.results[0])

Selected device: MPS:0
Bead radii table (A):
  1 residue/bead -> radius 3.200 A
  2 residue/bead -> radius 4.032 A
  3 residue/bead -> radius 4.615 A
  4 residue/bead -> radius 5.080 A
  5 residue/bead -> radius 5.472 A
Device selected: MPS:0
Sampling center-biased sphere centers, bead sizes (1..5 residues), and Monte Carlo crosslink probabilities...
Processed batch 15/157
Processed batch 30/157
Processed batch 45/157
Processed batch 60/157
Processed batch 75/157
Processed batch 90/157
Processed batch 105/157
Processed batch 120/157
Processed batch 135/157
Processed batch 150/157
Completed: generated 5000 configurations.
Generated rows: 5000
First row [center_d, r1, r2, L, res1, res2, prob]:
[0.0, -1618751.75, -1618751.75, 21.0, 3.0, 3.0, 1.0]


Sequential split keys equal?
False
[2891654438 1268651290] [3118485362 3383419422]

Sequentially sampled centers identical?
False

Levels1: [4 3 1 5 1 2 5 5]
Levels2: [5 3 2 4 1 1 3 3]

Manual batch probability min/max:
0.0 1.0
[0. 1. 0. 0. 0. 0. 1. 1.]
